In [1]:
import pandas as pd
import numpy as np

In [2]:
# df_site = pd.read_excel(r'Ethiopia DTM - SA R38_Dataset_Sharable without GPS.xlsx', sheet_name='Sites')

In [3]:
df_site = pd.read_excel(r'Ethiopia DTM - R39_Dataset_EC.xlsx')

In [4]:
# df_site_south = pd.concat([df_site, df_south_et])

In [5]:
# df_site= df_site_south

In [6]:
# df_site.to_excel('Merged data frame with south Ethiopia.xlsx')

In [7]:
df_site.columns.to_list()

['M-1658: Survey Date',
 'M-0489: Survey Round',
 'M-0445: Site ID',
 'M-0448: Site Name',
 '1.1.d.2: Site Alternate Name',
 '1.4.a.2: Is site open?',
 'M-0303: Region',
 'M-0304: Zone',
 'M-0305: Woreda',
 'M-0306: Kebele',
 'OCHA Region',
 'OCHA Region P-Code',
 'OCHA Zone',
 'OCHA Zone P-Code',
 'OCHA Woreda',
 'OCHA Woreda P-Code',
 '1.1.f.1: GPS: Longitude',
 '1.1.f.2: GPS: Latitude',
 '1.4.a.1: Site Open Date',
 'S-1835: Site Started',
 'xxxx: If reopened, when did it reopen?',
 'M-0337: Site Classification',
 'M-0342: Settlement/site type',
 'M-0342: If settlement/site type is other, Please specify',
 'S2302: If site type is collective center, what type of collective center?',
 'S2302: If the type of collective center is other, Please specify',
 '1.1.i.1: Is the site physically accessible by car?',
 'xxxx: If no, select all alternative options of physical accessibility',
 'xxxx: If no, select all alternative options of physical accessibility/Accessible by truck',
 'xxxx: If no, 

In [8]:
df_site['M-0342: Settlement/site type'].value_counts()

M-0342: Settlement/site type
Spontaneous camp/site    43
Host community           37
Collective center         7
Name: count, dtype: int64

In [9]:
df_site['Recoded_comp'] = df_site['M-0342: Settlement/site type'].replace({'Spontaneous camp/site': 'In Camp', 
                                                                'Collective center': 'In Camp', 
                                                                'Planned camp/site': 'In Camp',
                                                                'Host community': 'Out of Camp',
                                                                'Dispersed settlement':'Out of Camp'})

In [10]:
df_site['Recoded_comp'].value_counts()

Recoded_comp
In Camp        50
Out of Camp    37
Name: count, dtype: int64

In [11]:
df_zonal_level = df_site.groupby(['M-0303: Region','OCHA Zone'])['M-0309: Total Number of IDP HHs'].sum().reset_index()

In [12]:
zonal_level = df_site.groupby(['M-0303: Region','OCHA Zone','Recoded_comp'], dropna=False)['M-0309: Total Number of IDP HHs'].sum().reset_index()

In [13]:
df_zonal_level['M-0309: Total Number of IDP HHs'].sum()

np.int64(20306)

In [14]:
zonal_level 

,M-0303: Region,OCHA Zone,Recoded_comp,M-0309: Total Number of IDP HHs
0,South Ethiopia,Gofa,In Camp,545
1,South Ethiopia,Gofa,Out of Camp,301
2,South Ethiopia,Konso,In Camp,846
3,South Ethiopia,Konso,Out of Camp,1678
4,South Ethiopia,South Omo_Pas,In Camp,16936


In [15]:
df_zonal_cross =  pd.crosstab(index= [zonal_level['M-0303: Region'], zonal_level['OCHA Zone']],
                                 columns= zonal_level['Recoded_comp'],
                                     values=zonal_level['M-0309: Total Number of IDP HHs'],
                                 aggfunc='sum'
                                        ).reset_index()

In [16]:
df_zonal_cross

Recoded_comp,M-0303: Region,OCHA Zone,In Camp,Out of Camp
0,South Ethiopia,Gofa,545.0,301.0
1,South Ethiopia,Konso,846.0,1678.0
2,South Ethiopia,South Omo_Pas,16936.0,NaN


In [17]:
df_woreda_cross =  pd.crosstab(index= [df_site['M-0303: Region'], df_site['OCHA Zone'], df_site['OCHA Woreda']],
                                 columns= df_site['Recoded_comp'],
                                 values = df_site['M-0309: Total Number of IDP HHs'],
                                 aggfunc='sum'
                                        ).reset_index()

In [18]:
# Zo_exclusion = df_zonal_cross[(df_zonal_cross['In Camp'].notna()) & (df_zonal_cross['Out of Camp'].notna())]

In [19]:
# Woreda_pop = df_woreda_cross[(df_woreda_cross['In Camp'].notna()) & (df_woreda_cross['Out of Camp'].notna())]

In [20]:
# Woreda_pop.shape

In [21]:
# Zo_exclusion.columns

In [22]:
# df_woreda_cross['Out of Camp'].sum()

In [23]:
def calculate_sample_size(df, population_size_col):
    
    Z = 1.645  # Z-score for 90% confidence level
    P = 0.5   # Proportion (assumed maximum variability)
    d = 0.1  # Margin of error
    deff = 1.5  # Design Effect

    def sample_size_formula(N):
        
        numerator = (Z**2) * P * N * (1 - P) * deff
        denominator = (d**2) * (N - 1) + (Z**2) * P * (1 - P)

        
        if denominator == 0:
            return np.nan
        
        
        sample_size = np.ceil(numerator / denominator)
        return min(sample_size, N)  

   
    df['Sample_Size_incomp'] = df['In Camp'].apply(sample_size_formula)
    df['Sample_Size_outofcamp'] = df['Out of Camp'].apply(sample_size_formula)
    
    return df



df = pd.DataFrame(df_zonal_cross)


df = calculate_sample_size(df, 'Population_Size')


In [24]:
df['_reserve_samples_incomp'] = ((df['Sample_Size_incomp']*20)/100).round(0)

In [25]:
df['_reserve_samples_Outofcomp'] = ((df['Sample_Size_outofcamp']*20)/100).round(0)

In [26]:
df['In ccmp Sample size'] = df[['_reserve_samples_incomp', 'Sample_Size_incomp']].sum(axis=1)

In [27]:
df['Out of ccmp Sample size'] = df[['_reserve_samples_Outofcomp', 'Sample_Size_outofcamp']].sum(axis=1)

In [28]:
df['Out of ccmp Sample size'].sum()

np.float64(219.0)

In [29]:
df['In ccmp Sample size'].sum()

np.float64(345.0)

In [30]:
7402+5761    ##samples shared with Boss =13725

13163

In [31]:
# df.to_excel('dataframe_sample.xlsx')

In [32]:
# df['Number of clusters to be selected'] = df['In ccmp Sample size'] / 17

In [33]:
df_sample_incomp = df[['M-0303: Region', 'OCHA Zone','In ccmp Sample size','In Camp']] 

In [34]:
df_sample_incomp['Number of clusters to be selected'] = (df_sample_incomp['In ccmp Sample size'] / 17).round(0)

C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\2566396676.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample_incomp['Number of clusters to be selected'] = (df_sample_incomp['In ccmp Sample size'] / 17).round(0)


In [35]:
df_sample_ouofcomp = df[['M-0303: Region', 'OCHA Zone','Out of ccmp Sample size','Out of Camp']] 

In [36]:
df_sample_ouofcomp['Number of clusters to be selected'] = (df_sample_ouofcomp['Out of ccmp Sample size'] / 17).round(0)

C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\3413005135.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample_ouofcomp['Number of clusters to be selected'] = (df_sample_ouofcomp['Out of ccmp Sample size'] / 17).round(0)


In [37]:
df_sample_ouofcomp['Number of clusters to be selected'].describe()

count    3.000000
mean     4.333333
std      3.785939
min      0.000000
25%      3.000000
50%      6.000000
75%      6.500000
max      7.000000
Name: Number of clusters to be selected, dtype: float64

In [38]:
df_site2 = df_site[['M-0303: Region','OCHA Zone','OCHA Woreda', 'M-0445: Site ID', 'M-0448: Site Name','M-0309: Total Number of IDP HHs', 'Recoded_comp']]

In [39]:
df_incomp_sites = df_site2[df_site2['Recoded_comp'] == 'In Camp']
df_outofcomp_sites = df_site2[df_site2['Recoded_comp'] == 'Out of Camp']

#### Sampling At Site level 

### In Camp

In [40]:
# In-camp sampling 

# Initialize output columns
df_incomp_sites.loc[:, 'Site Status of Selection'] = 'Not selected'
df_incomp_sites.loc[:, 'Sample to be Interviewed per Selected Site'] = 0
df_incomp_sites.loc[:, 'clusters and sample summary'] = ''

# Sort by zone and descending population
df_incomp_sites = df_incomp_sites.sort_values(
    ['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False]
)

# Process each OCHA Zone
for _, row in df_sample_incomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['In ccmp Sample size'])

    # Filter the sites for this zone
    zone_sites = df_incomp_sites[df_incomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine number of sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top-N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark selected
    df_incomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Sum of available population in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Use actual population if sample size is too high
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Integer division and remainder to ensure no rounding issues
        base_sample = final_sample_size // n_to_select
        remainder = final_sample_size % n_to_select

        # Assign base sample to all selected sites
        df_incomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = base_sample

        # Distribute the remainder to the first 'remainder' sites
        if remainder > 0:
            extra_indices = selected_indices[:remainder]
            df_incomp_sites.loc[extra_indices, 'Sample to be Interviewed per Selected Site'] += 1

        # Create and assign summary string to all rows in the zone
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_incomp_sites.loc[df_incomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# Optional preview
# print(df_incomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                        'Site Status of Selection', 'Sample to be Interviewed per Selected Site',
#                        'clusters and sample summary']])


C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\593697085.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_incomp_sites.loc[:, 'Site Status of Selection'] = 'Not selected'
C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\593697085.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_incomp_sites.loc[:, 'Sample to be Interviewed per Selected Site'] = 0
C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\593697085.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

### Out of camp 

In [41]:
# Initialize output columns safely
df_outofcomp_sites.loc[:, 'Site Status of Selection'] = 'Not selected'
df_outofcomp_sites.loc[:, 'Sample to be Interviewed per Selected Site'] = 0
df_outofcomp_sites.loc[:, 'clusters and sample summary'] = ''

# Sort by zone and descending population
df_outofcomp_sites = df_outofcomp_sites.sort_values(
    ['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False]
)

# Process each OCHA Zone
for _, row in df_sample_ouofcomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['Out of ccmp Sample size'])

    # Filter the sites for this zone
    zone_sites = df_outofcomp_sites[df_outofcomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine number of sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top-N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark selected
    df_outofcomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Sum of available population in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Use actual population if sample size is too high
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Integer division and remainder to avoid rounding differences
        base_sample = final_sample_size // n_to_select
        remainder = final_sample_size % n_to_select

        # Assign base sample to all selected sites
        df_outofcomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = base_sample

        # Distribute remainder to the first 'remainder' sites
        if remainder > 0:
            extra_indices = selected_indices[:remainder]
            df_outofcomp_sites.loc[extra_indices, 'Sample to be Interviewed per Selected Site'] += 1

        # Create and assign summary string to all rows in the zone
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_outofcomp_sites.loc[df_outofcomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# Optional preview
# print(df_outofcomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                           'Site Status of Selection', 'Sample to be Interviewed per Selected Site',
#                           'clusters and sample summary']])


C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\3613335650.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_outofcomp_sites.loc[:, 'Site Status of Selection'] = 'Not selected'
C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\3613335650.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_outofcomp_sites.loc[:, 'Sample to be Interviewed per Selected Site'] = 0
C:\Users\aadma\AppData\Local\Temp\ipykernel_18936\3613335650.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

In [42]:
with pd.ExcelWriter('Sampled_sites_Incomp and Out of Camp_r39.xlsx', engine='xlsxwriter') as writer:
    df_incomp_sites.to_excel(writer, sheet_name='In-Camp Sites', index=False)
    df_outofcomp_sites.to_excel(writer, sheet_name='Out-of-Camp Sites', index=False)


In [185]:
# Initialize output columns
df_incomp_sites['Site Status of Selection'] = 'Not selected'
df_incomp_sites['Sample to be Interviewed per Selected Site'] = 0
df_incomp_sites['clusters and sample summary'] = ''

# Sort sites by zone and descending IDP HHs
df_incomp_sites = df_incomp_sites.sort_values(['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False])

# Process each OCHA Zone
for _, row in df_sample_incomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['In ccmp Sample size'])

    # Filter the sites for this zone
    zone_sites = df_incomp_sites[df_incomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine number of sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark selected
    df_incomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Sum of available population in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Use actual population if sample size is too high
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Calculate sample per site (rounded)
        sample_per_site = round(final_sample_size / n_to_select)

        # Assign to selected sites
        df_incomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = sample_per_site

        # Create summary string
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_incomp_sites.loc[df_incomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# Preview result
# print(df_incomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
                       # 'sampled site', 'sample size per site', 'clusters and sample summary']])


In [186]:
# Initialize output columns
df_outofcomp_sites['Site Status of Selection'] = 'Not selected'
df_outofcomp_sites['Sample to be Interviewed per Selected Site'] = 0
df_outofcomp_sites['clusters and sample summary'] = ''

# Sort by size within zone
df_outofcomp_sites = df_outofcomp_sites.sort_values(
    ['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False]
)

# Loop through each zone in sampling plan
for _, row in df_sample_ouofcomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    original_sample_size = int(row['Out of ccmp Sample size'])

    # Filter all sites in this zone
    zone_sites = df_outofcomp_sites[df_outofcomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Determine how many sites to select
    n_to_select = min(n_clusters, available_sites)

    # Select top-N sites
    selected_sites = zone_sites.head(n_to_select)
    selected_indices = selected_sites.index

    # Mark them as selected
    df_outofcomp_sites.loc[selected_indices, 'Site Status of Selection'] = 'Selected'

    if n_to_select > 0:
        # Calculate the total IDP HHs in selected sites
        total_idp_hhs = selected_sites['M-0309: Total Number of IDP HHs'].sum()

        # Adjust sample size if needed
        final_sample_size = min(original_sample_size, total_idp_hhs)

        # Calculate equal share per site (rounded to 0 decimals)
        sample_per_site = round(final_sample_size / n_to_select)

        # Assign sample size per site
        df_outofcomp_sites.loc[selected_indices, 'Sample to be Interviewed per Selected Site'] = sample_per_site

        # Assign summary string to all sites in this zone
        summary_text = f'Clusters: {n_to_select}, Sample: {final_sample_size}'
        df_outofcomp_sites.loc[df_outofcomp_sites['OCHA Zone'] == zone, 'Clusters number per zone and Sample summary'] = summary_text

# Optional: Preview result
# print(df_outofcomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                           'sampled site', 'sample size per site', 'clusters and sample summary']])


In [ ]:

# Initialize output columns
df_outofcomp_sites['sampled site'] = 'not selected'
df_outofcomp_sites['sample size per site'] = 0
df_outofcomp_sites['clusters and sample summary'] = ''

# Sort by size within zone
df_outofcomp_sites = df_outofcomp_sites.sort_values(['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False])

# Loop through each zone
for _, row in df_sample_ouofcomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    total_sample_size = int(row['Out of ccmp Sample size'])

    # Get all sites in the zone
    zone_sites = df_outofcomp_sites[df_outofcomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Decide how many to select
    n_to_select = min(n_clusters, available_sites)

    # Get indices of selected sites
    selected_indices = zone_sites.head(n_to_select).index

    # Mark as selected
    df_outofcomp_sites.loc[selected_indices, 'sampled site'] = 'selected'

    # Calculate and assign sample size per site (rounded)
    if n_to_select > 0:
        sample_per_site = round(total_sample_size / n_to_select)
        df_outofcomp_sites.loc[selected_indices, 'sample size per site'] = sample_per_site

        # Create summary string
        summary_text = f'Clusters: {n_to_select}, Sample: {total_sample_size}'
        df_outofcomp_sites.loc[df_outofcomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# # Preview result
# print(df_outofcomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                        'sampled site', 'sample size per site', 'clusters and sample summary']])

In [ ]:
# Initialize output columns
df_incomp_sites['sampled site'] = 'not selected'
df_incomp_sites['sample size per site'] = 0
df_incomp_sites['clusters and sample summary'] = ''

# Sort by size within zone
df_incomp_sites = df_incomp_sites.sort_values(['OCHA Zone', 'M-0309: Total Number of IDP HHs'], ascending=[True, False])

# Loop through each zone
for _, row in df_sample_incomp.iterrows():
    zone = row['OCHA Zone']
    n_clusters = int(row['Number of clusters to be selected'])
    total_sample_size = int(row['In ccmp Sample size'])

    # Get all sites in the zone
    zone_sites = df_incomp_sites[df_incomp_sites['OCHA Zone'] == zone]
    available_sites = len(zone_sites)

    # Decide how many to select
    n_to_select = min(n_clusters, available_sites)

    # Get indices of selected sites
    selected_indices = zone_sites.head(n_to_select).index

    # Mark as selected
    df_incomp_sites.loc[selected_indices, 'sampled site'] = 'selected'

    # Calculate and assign sample size per site (rounded)
    if n_to_select > 0:
        sample_per_site = round(total_sample_size / n_to_select)
        df_incomp_sites.loc[selected_indices, 'sample size per site'] = sample_per_site

        # Create summary string
        summary_text = f'Clusters: {n_to_select}, Sample: {total_sample_size}'
        df_incomp_sites.loc[df_incomp_sites['OCHA Zone'] == zone, 'clusters and sample summary'] = summary_text

# # Preview result
# print(df_incomp_sites[['M-0303: Region', 'OCHA Zone', 'OCHA Woreda', 'M-0445: Site ID',
#                        'sampled site', 'sample size per site', 'clusters and sample summary']])


In [30]:
df_sample_melted =  df.melt(id_vars=['M-0303: Region', 'OCHA Zone'], 
                                        value_vars=['Sample_Size_incomp', 'Sample_Size_outofcamp'],
                                       value_name='Sample Size',
                                       var_name='Pop_group')
# df_sample_melted = df_sample_melted[df_sample_melted['Sample Size'] != 0]

In [31]:
df_sample_melted 

,M-0303: Region,OCHA Zone,Pop_group,Sample Size
0,Afar,Awsi /Zone 1,Sample_Size_incomp,97.0
1,Afar,Fanti /Zone 4,Sample_Size_incomp,NaN
2,Afar,Gabi /Zone 3,Sample_Size_incomp,100.0
3,Afar,Hari /Zone 5,Sample_Size_incomp,70.0
4,Afar,Kilbati /Zone2,Sample_Size_incomp,99.0
...,...,...,...,...
145,Tigray,Eastern,Sample_Size_outofcamp,101.0
146,Tigray,Mekelle,Sample_Size_outofcamp,101.0
147,Tigray,North Western,Sample_Size_outofcamp,101.0
148,Tigray,South Eastern,Sample_Size_outofcamp,101.0


In [32]:
df_sample_melted['camp type'] = df_sample_melted['Pop_group'].replace({'Out of ccmp Sample size': 'Out of Camp',
                                                          'In ccmp Sample size':'In Camp'}) 

In [33]:
df_sample_melted

,M-0303: Region,OCHA Zone,Pop_group,Sample Size,camp type
0,Afar,Awsi /Zone 1,Sample_Size_incomp,97.0,Sample_Size_incomp
1,Afar,Fanti /Zone 4,Sample_Size_incomp,NaN,Sample_Size_incomp
2,Afar,Gabi /Zone 3,Sample_Size_incomp,100.0,Sample_Size_incomp
3,Afar,Hari /Zone 5,Sample_Size_incomp,70.0,Sample_Size_incomp
4,Afar,Kilbati /Zone2,Sample_Size_incomp,99.0,Sample_Size_incomp
...,...,...,...,...,...
145,Tigray,Eastern,Sample_Size_outofcamp,101.0,Sample_Size_outofcamp
146,Tigray,Mekelle,Sample_Size_outofcamp,101.0,Sample_Size_outofcamp
147,Tigray,North Western,Sample_Size_outofcamp,101.0,Sample_Size_outofcamp
148,Tigray,South Eastern,Sample_Size_outofcamp,101.0,Sample_Size_outofcamp


In [44]:
df_site2 = df_site[['M-0303: Region','OCHA Zone','OCHA Woreda', 'M-0445: Site ID', 'M-0448: Site Name','M-0309: Total Number of IDP HHs', 'Recoded_comp']]

In [35]:
df_site3 = pd.crosstab(index= [df_site2['M-0303: Region'],df_site2['OCHA Zone']], 
                       columns=df_site2['Recoded_comp'], 
                       values=df_site2['M-0445: Site ID'],
                      aggfunc='count').reset_index()

In [36]:
df_site3.columns

Index(['M-0303: Region', 'OCHA Zone', 'In Camp', 'Out of Camp'], dtype='object', name='Recoded_comp')

In [37]:
df_site_final_mel=  df_site3.melt(id_vars=['M-0303: Region', 'OCHA Zone'], 
                                        value_vars=['In Camp', 'Out of Camp'],
                                       var_name='camp type',
                                       value_name='Total number of IDP camps')

In [38]:
df_site2

,M-0303: Region,OCHA Zone,OCHA Woreda,M-0445: Site ID,M-0448: Site Name,M-0309: Total Number of IDP HHs,Recoded_comp
0,Oromia,Horo Gudru Wellega,Choman Guduru,OR2166,Ayale Market Area,60,Out of Camp
1,Afar,Gabi /Zone 3,Awash Fantale,AF477,Gomedle,250,In Camp
2,Afar,Gabi /Zone 3,Awash Fantale,AF478,Elahabure,210,In Camp
3,Amhara,North Shewa (AM),Molale town,AM1052,Kobil,102,Out of Camp
4,Amhara,North Shewa (AM),Mehal Meda town,AM545,Mehal Meda kebele Sost,94,Out of Camp
...,...,...,...,...,...,...,...
2677,Tigray,Central,Emba Sieneti,TG666,Adergeto,96,Out of Camp
2678,Tigray,Central,Aheferom,TG653,Maikeyahat,51,Out of Camp
2679,Tigray,Central,Rama Adi Arbaete,TG707,Maiweini,59,Out of Camp
2680,Tigray,Central,Adwa,TG313,Yeha,106,Out of Camp


In [39]:
df_site_merged = df_sample_melted.merge(df_site_final_mel, on=['M-0303: Region','OCHA Zone','camp type'], how='inner' )

In [40]:
# df_site_merged.to_excel(r'Main Final 150 samples.xlsx')

In [41]:
df_site2

,M-0303: Region,OCHA Zone,OCHA Woreda,M-0445: Site ID,M-0448: Site Name,M-0309: Total Number of IDP HHs,Recoded_comp
0,Oromia,Horo Gudru Wellega,Choman Guduru,OR2166,Ayale Market Area,60,Out of Camp
1,Afar,Gabi /Zone 3,Awash Fantale,AF477,Gomedle,250,In Camp
2,Afar,Gabi /Zone 3,Awash Fantale,AF478,Elahabure,210,In Camp
3,Amhara,North Shewa (AM),Molale town,AM1052,Kobil,102,Out of Camp
4,Amhara,North Shewa (AM),Mehal Meda town,AM545,Mehal Meda kebele Sost,94,Out of Camp
...,...,...,...,...,...,...,...
2677,Tigray,Central,Emba Sieneti,TG666,Adergeto,96,Out of Camp
2678,Tigray,Central,Aheferom,TG653,Maikeyahat,51,Out of Camp
2679,Tigray,Central,Rama Adi Arbaete,TG707,Maiweini,59,Out of Camp
2680,Tigray,Central,Adwa,TG313,Yeha,106,Out of Camp


In [55]:
df_merged_pps_site2 = df_sa_pps.merge(df_site2[['M-0445: Site ID', 'Recoded_comp','M-0309: Total Number of IDP HHs']], on='M-0445: Site ID', how='inner' )

In [56]:
df_merged_pps_site2

,M-0303: Region,OCHA Zone,OCHA Woreda,M-0445: Site ID,M-0448: Site Name,M-0309: Total Number of IDP HHs_x,pps_rank,percentage_of_HHs,Recoded_comp,M-0309: Total Number of IDP HHs_y
0,Oromia,Horo Gudru Wellega,Choman Guduru,OR2166,Ayale Market Area,60,4,2.17,Out of Camp,60
1,Afar,Gabi /Zone 3,Awash Fantale,AF477,Gomedle,250,30,3.19,In Camp,250
2,Afar,Gabi /Zone 3,Awash Fantale,AF478,Elahabure,210,26,2.68,In Camp,210
3,Amhara,North Shewa (AM),Molale town,AM1052,Kobil,102,35,0.75,Out of Camp,102
4,Amhara,North Shewa (AM),Mehal Meda town,AM545,Mehal Meda kebele Sost,94,32,0.70,Out of Camp,94
...,...,...,...,...,...,...,...,...,...,...
2677,Tigray,Central,Emba Sieneti,TG666,Adergeto,96,110,0.19,Out of Camp,96
2678,Tigray,Central,Aheferom,TG653,Maikeyahat,51,55,0.10,Out of Camp,51
2679,Tigray,Central,Rama Adi Arbaete,TG707,Maiweini,59,60,0.11,Out of Camp,59
2680,Tigray,Central,Adwa,TG313,Yeha,106,122,0.21,Out of Camp,106


In [57]:
df_site4 = pd.crosstab(index= [df_merged_pps_site2['M-0303: Region'],df_merged_pps_site2['OCHA Zone']], 
                       columns=df_merged_pps_site2['Recoded_comp'], 
                       values=df_merged_pps_site2['M-0445: Site ID'],
                      aggfunc='count').reset_index()

In [58]:
df_site3

Recoded_comp,M-0303: Region,OCHA Zone,In Camp,Out of Camp
0,Afar,Awsi /Zone 1,11.0,6.0
1,Afar,Fanti /Zone 4,NaN,2.0
2,Afar,Gabi /Zone 3,39.0,2.0
3,Afar,Hari /Zone 5,2.0,NaN
4,Afar,Kilbati /Zone2,9.0,4.0
...,...,...,...,...
70,Tigray,Eastern,8.0,128.0
71,Tigray,Mekelle,16.0,31.0
72,Tigray,North Western,46.0,117.0
73,Tigray,South Eastern,2.0,73.0


In [50]:
df_sample_melted.columns

Index(['M-0303: Region', 'OCHA Zone', 'Pop_group', 'Sample Size', 'camp type'], dtype='object')

In [51]:
df_sam = pd.crosstab(index= [df_sample_melted['M-0303: Region'],df_sample_melted['OCHA Zone']], 
                           columns=df_sample_melted['Pop_group'], 
                       values=df_sample_melted['Sample Size'],
                      aggfunc='sum').reset_index()

In [52]:
df_sam 

Pop_group,M-0303: Region,OCHA Zone,Sample_Size_incomp,Sample_Size_outofcamp
0,Afar,Awsi /Zone 1,97.0,95.0
1,Afar,Fanti /Zone 4,0.0,55.0
2,Afar,Gabi /Zone 3,100.0,84.0
3,Afar,Hari /Zone 5,70.0,0.0
4,Afar,Kilbati /Zone2,99.0,88.0
...,...,...,...,...
70,Tigray,Eastern,94.0,101.0
71,Tigray,Mekelle,101.0,101.0
72,Tigray,North Western,101.0,101.0
73,Tigray,South Eastern,70.0,101.0


In [53]:
df_mer = df_sam.merge(df_site4, on=['M-0303: Region', 'OCHA Zone'], how='inner' )

In [109]:
# df_mer.to_excel('New sample and clusters merged.xlsx')

In [59]:
df_mer

,M-0303: Region,OCHA Zone,Sample_Size_incomp,Sample_Size_outofcamp,In Camp,Out of Camp
0,Afar,Awsi /Zone 1,97.0,95.0,11.0,6.0
1,Afar,Fanti /Zone 4,0.0,55.0,NaN,2.0
2,Afar,Gabi /Zone 3,100.0,84.0,39.0,2.0
3,Afar,Hari /Zone 5,70.0,0.0,2.0,NaN
4,Afar,Kilbati /Zone2,99.0,88.0,9.0,4.0
...,...,...,...,...,...,...
70,Tigray,Eastern,94.0,101.0,8.0,128.0
71,Tigray,Mekelle,101.0,101.0,16.0,31.0
72,Tigray,North Western,101.0,101.0,46.0,117.0
73,Tigray,South Eastern,70.0,101.0,2.0,73.0


In [173]:
# df['Total sample size reserves']= (df['sample']*20)/100

In [174]:
# df['Sample_Size_outofcamp'].sum()

In [175]:
# df['Total Sample size'] = df[['Sample_Size_incomp','Total sample size reserves']].sum(axis=1)

### Determining the number of Clusters

In [176]:
# df.to_excel('Final _DE_1.5_nei.xlsx')

In [177]:
df_sample_size.columns

Index(['M-0303: Region', 'OCHA Zone', 'In Camp', 'Out of Camp',
       'Sample_Size_incomp', 'Sample_Size_outofcamp', 'Response rate 10%',
       'Response rate 20%', 'Total population @10% Nonresponse rate',
       'Total population @20% Nonresponse rate'],
      dtype='object')

In [203]:
df_melted_group =  df_sample_melted.groupby(['M-0303: Region', 'OCHA Zone','Pop_group']) ['Sample Size'].sum().reset_index()

In [204]:
# df['reserve']=  df['Sample_Size_outofcamp'] 

In [205]:
df_melted_group

,M-0303: Region,OCHA Zone,Pop_group,Sample Size
0,Afar,Awsi /Zone 1,In ccmp Sample size,116.0
1,Afar,Awsi /Zone 1,Out of ccmp Sample size,114.0
2,Afar,Fanti /Zone 4,Out of ccmp Sample size,66.0
3,Afar,Gabi /Zone 3,In ccmp Sample size,120.0
4,Afar,Gabi /Zone 3,Out of ccmp Sample size,101.0
...,...,...,...,...
121,Tigray,North Western,Out of ccmp Sample size,121.0
122,Tigray,South Eastern,In ccmp Sample size,84.0
123,Tigray,South Eastern,Out of ccmp Sample size,121.0
124,Tigray,Southern,In ccmp Sample size,59.0


In [208]:
# df_melted_group.to_excel('Melted_grouped.xlsx')


In [224]:
df_site2 = df_site[['M-0303: Region','OCHA Zone','OCHA Woreda','Recoded_comp', 'M-0445: Site ID', 'M-0448: Site Name','M-0309: Total Number of IDP HHs', 'Recoded_comp']]

In [225]:
df_site2 

,M-0303: Region,OCHA Zone,OCHA Woreda,Recoded_comp,M-0445: Site ID,M-0448: Site Name,M-0309: Total Number of IDP HHs,Recoded_comp
0,Oromia,Horo Gudru Wellega,Choman Guduru,Out of Camp,OR2166,Ayale Market Area,60,Out of Camp
1,Afar,Gabi /Zone 3,Awash Fantale,In Camp,AF477,Gomedle,250,In Camp
2,Afar,Gabi /Zone 3,Awash Fantale,In Camp,AF478,Elahabure,210,In Camp
3,Amhara,North Shewa (AM),Molale town,Out of Camp,AM1052,Kobil,102,Out of Camp
4,Amhara,North Shewa (AM),Mehal Meda town,Out of Camp,AM545,Mehal Meda kebele Sost,94,Out of Camp
...,...,...,...,...,...,...,...,...
2677,Tigray,Central,Emba Sieneti,Out of Camp,TG666,Adergeto,96,Out of Camp
2678,Tigray,Central,Aheferom,Out of Camp,TG653,Maikeyahat,51,Out of Camp
2679,Tigray,Central,Rama Adi Arbaete,Out of Camp,TG707,Maiweini,59,Out of Camp
2680,Tigray,Central,Adwa,Out of Camp,TG313,Yeha,106,Out of Camp


In [226]:

df_site2['pps_rank'] = df_site2.groupby('OCHA Zone')['M-0309: Total Number of IDP HHs'].rank(ascending=True, method='first').astype(int)


C:\Users\aadmassie\AppData\Local\Temp\ipykernel_29668\1567180409.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_site2['pps_rank'] = df_site2.groupby('OCHA Zone')['M-0309: Total Number of IDP HHs'].rank(ascending=True, method='first').astype(int)


In [227]:
# df_site.to_excel('final for PPs based on zonal.xlsx')

In [228]:
df_site2['percentage_of_HHs'] = df_site2.groupby('OCHA Zone')['M-0309: Total Number of IDP HHs'].apply(
    lambda x: (x / x.sum() * 100).round(2)).reset_index(level=0, drop=True)

C:\Users\aadmassie\AppData\Local\Temp\ipykernel_29668\822062190.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_site2['percentage_of_HHs'] = df_site2.groupby('OCHA Zone')['M-0309: Total Number of IDP HHs'].apply(


In [229]:
# df_site2.to_excel('final for PPs based on zonal_pps.xlsx')

In [230]:
df_site2

,M-0303: Region,OCHA Zone,OCHA Woreda,Recoded_comp,M-0445: Site ID,M-0448: Site Name,M-0309: Total Number of IDP HHs,Recoded_comp,pps_rank,percentage_of_HHs
0,Oromia,Horo Gudru Wellega,Choman Guduru,Out of Camp,OR2166,Ayale Market Area,60,Out of Camp,4,2.17
1,Afar,Gabi /Zone 3,Awash Fantale,In Camp,AF477,Gomedle,250,In Camp,30,3.19
2,Afar,Gabi /Zone 3,Awash Fantale,In Camp,AF478,Elahabure,210,In Camp,26,2.68
3,Amhara,North Shewa (AM),Molale town,Out of Camp,AM1052,Kobil,102,Out of Camp,35,0.75
4,Amhara,North Shewa (AM),Mehal Meda town,Out of Camp,AM545,Mehal Meda kebele Sost,94,Out of Camp,32,0.70
...,...,...,...,...,...,...,...,...,...,...
2677,Tigray,Central,Emba Sieneti,Out of Camp,TG666,Adergeto,96,Out of Camp,110,0.19
2678,Tigray,Central,Aheferom,Out of Camp,TG653,Maikeyahat,51,Out of Camp,55,0.10
2679,Tigray,Central,Rama Adi Arbaete,Out of Camp,TG707,Maiweini,59,Out of Camp,60,0.11
2680,Tigray,Central,Adwa,Out of Camp,TG313,Yeha,106,Out of Camp,122,0.21


In [231]:
# df_site2.to_excel('Cluster_size_final for PPs based on zonal_pps.xlsx')

#### Determining the number of Clusters 

In [117]:
# df_site_cluster =  df_site2.groupby(['M-0303: Region', 'OCHA Zone','Recoded_comp']) ['M-0445: Site ID'].count().reset_index()

In [217]:
df_site_cluster

,M-0303: Region,OCHA Zone,Recoded_comp,M-0445: Site ID
0,Afar,Awsi /Zone 1,In Camp,11
1,Afar,Awsi /Zone 1,Out of Camp,6
2,Afar,Fanti /Zone 4,Out of Camp,2
3,Afar,Gabi /Zone 3,In Camp,39
4,Afar,Gabi /Zone 3,Out of Camp,2
...,...,...,...,...
121,Tigray,North Western,Out of Camp,117
122,Tigray,South Eastern,In Camp,2
123,Tigray,South Eastern,Out of Camp,73
124,Tigray,Southern,In Camp,2


In [218]:
df_site_cluster.drop_duplicates(keep='first')

,M-0303: Region,OCHA Zone,Recoded_comp,M-0445: Site ID
0,Afar,Awsi /Zone 1,In Camp,11
1,Afar,Awsi /Zone 1,Out of Camp,6
2,Afar,Fanti /Zone 4,Out of Camp,2
3,Afar,Gabi /Zone 3,In Camp,39
4,Afar,Gabi /Zone 3,Out of Camp,2
...,...,...,...,...
121,Tigray,North Western,Out of Camp,117
122,Tigray,South Eastern,In Camp,2
123,Tigray,South Eastern,Out of Camp,73
124,Tigray,Southern,In Camp,2


In [219]:
df_final = df_site_cluster.merge(df_melted_group, on=['M-0303: Region', 'OCHA Zone'], how='right')

In [220]:
df_final['Sample Size'].sum()

23933.0

In [128]:
df_44.shape

(252,)

In [196]:
df_final.to_excel('Main for checks.xlsx')